In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Carrega o dataset bruto
df = pd.read_csv("../data/raw/telco_churn.csv")

# Remove o identificador - não é uma feature preditiva
df = df.drop(columns=['customerID'])

# Corrige TotalCharges (mesmo tratamento identificado na EDA)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)

# Transforma o alvo em binário
df['Churn'] = df['Churn'].map({'No': 0, 'Yes': 1})

# Separa features e alvo
X = df.drop(columns=['Churn'])
y = df['Churn']

# Split treino/teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Treino:", X_train.shape, "| Teste:", X_test.shape)

Treino: (5634, 19) | Teste: (1409, 19)


In [6]:
# Identifica automaticamente quais colunas são numéricas e quais são categóricas
numeric_features = X.select_dtypes(
    include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

print("Numéricas:", numeric_features)
print("\nCategóricas:", categorical_features)

Numéricas: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

Categóricas: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


C:\Users\denar\AppData\Local\Temp\ipykernel_34384\3719368170.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(include=['object']).columns.tolist()


In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score, classification_report

# Pré-processamento: normaliza numéricas, codifica categóricas
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

# Pipeline completo: pré-processamento + modelo
baseline_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000))
])

# Treina o modelo (o pipeline cuida de aplicar o pré-processamento automaticamente)
baseline_pipeline.fit(X_train, y_train)

# Faz previsões no conjunto de teste
y_pred = baseline_pipeline.predict(X_test)
y_proba = baseline_pipeline.predict_proba(X_test)[:, 1]

# Avalia com as métricas definidas no ML Canvas
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)

print(f"F1-score: {f1:.4f}")
print(f"AUC-ROC: {auc:.4f}")
print("\nRelatório completo:")
print(classification_report(y_test, y_pred,
      target_names=['No Churn', 'Churn']))

F1-score: 0.6040
AUC-ROC: 0.8421

Relatório completo:
              precision    recall  f1-score   support

    No Churn       0.85      0.89      0.87      1035
       Churn       0.66      0.56      0.60       374

    accuracy                           0.81      1409
   macro avg       0.75      0.73      0.74      1409
weighted avg       0.80      0.81      0.80      1409

